## Irish Property Prices - Exploratory Analysis

Notebook 3 of 4. Explores the cleaned sales and BER data to establish price trends, geographic variation and the relationships that inform the modelling stage.

Two analysis windows are used throughout. Full history covers all 750,660 sales from 2010 to present and is used for trend analysis at county level. Eircode era covers the 228,905 sales from 2021 onward that carry a valid Eircode, and is used wherever routing key granularity is required.

In [ ]:
# Setup 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROCESSED = Path("../data/processed")

# consistent styling across the notebook
NAVY  = "#1f4e79"
RED   = "#c00000"
GREEN = "#4a9e6b"
AMBER = "#e0a030"
BG    = "#fafafa"
GRID  = "#e8e8e8"

def style_axis(ax):
    ax.set_facecolor(BG)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    return ax

def euro(ax, axis="y"):
    fmt = mticker.FuncFormatter(lambda v, _: f"€{v:,.0f}")
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(fmt)

In [ ]:
# Load cleaned data 

df = pd.read_parquet(PROCESSED / "ppr_with_ber.parquet")

df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M")

print(f"Loaded {len(df):,} sales")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# Define the two analysis windows 
# full history for trend analysis, Eircode era for anything
# requiring routing key granularity

full = df.copy()
eircode_era = df[(df["year"] >= 2021) & df["routing_key"].notna()].copy()

print(f"Full history:  {len(full):,} sales, {full['year'].min()} to {full['year'].max()}")
print(f"Eircode era:   {len(eircode_era):,} sales, {eircode_era['year'].min()} to {eircode_era['year'].max()}")
print(f"Routing keys:  {eircode_era['routing_key'].nunique()}")

## 1. National price trend

Monthly medians are plotted alongside monthly transaction volume, since the two behave differently and volume affects how much weight to place on any individual month.

In [ ]:
# Monthly national median price 

monthly = full.groupby("month").agg(
    median_price=("price_incl_vat", "median"),
    mean_price=("price_incl_vat", "mean"),
    n_sales=("price_incl_vat", "size"),
)
monthly.index = monthly.index.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                          gridspec_kw={"height_ratios": [2, 1]})
fig.patch.set_facecolor(BG)

ax = style_axis(axes[0])
ax.plot(monthly.index, monthly["median_price"], color=NAVY, linewidth=1.5, label="Median")
ax.plot(monthly.index, monthly["mean_price"], color=RED, linewidth=1.0,
        linestyle="--", alpha=0.7, label="Mean")
euro(ax)
ax.set_ylabel("Sale price")
ax.set_title("Irish residential property prices, 2010 to present",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

ax = style_axis(axes[1])
ax.bar(monthly.index, monthly["n_sales"], width=25, color=NAVY, alpha=0.6)
ax.set_ylabel("Sales per month")
ax.set_xlabel("Date")

plt.tight_layout()
plt.show()

In [ ]:
# Annual summary

annual = full.groupby("year").agg(
    n_sales=("price_incl_vat", "size"),
    median=("price_incl_vat", "median"),
    mean=("price_incl_vat", "mean"),
    p25=("price_incl_vat", lambda s: s.quantile(0.25)),
    p75=("price_incl_vat", lambda s: s.quantile(0.75)),
).round(0)

annual["yoy_pct"] = (annual["median"].pct_change() * 100).round(1)
print(annual.to_string())

The series traces the full Irish property cycle. Prices fall from a median of €215,650 in 2010 to a trough of €149,796 in 2013, a decline of 31%, before recovering steadily to €390,440 by 2026 - a rise of 161% from the bottom.

Growth peaks at 15.8% in 2017, then slows sharply to 3.6% in 2019 and 0.6% in 2020. From 2021 onward growth settles into a consistent 8-10% band.

The mean sits consistently above the median by €30,000 to €40,000, and the gap widens over time, indicating the upper end of the market has pulled away faster than the middle.

2026 covers seven months only and contains no December, which is consistently the highest-volume month. Its figures are not directly comparable to full years.

## 2. Seasonality

Transaction volume is heavily seasonal. December runs 41% above the monthly average and January 29% below - a spread of nearly two to one between the busiest and quietest months. Every one of the ten highest-volume months in the dataset is a December.

December 2014 is the single busiest month at 7,253 sales, with a median of €144,905 against an annual median of €155,000. This coincides with the introduction of the Central Bank's macroprudential mortgage lending rules in January 2015, and is consistent with a rush of lower-value completions ahead of the deadline.

In [ ]:
print(full[full["year"] == 2026].groupby("month").size().to_string())

In [ ]:
top_months = monthly.nlargest(10, "n_sales")[["n_sales", "median_price"]]
print(top_months.to_string())

# what drove the largest month?
peak = monthly["n_sales"].idxmax().to_period("M")
spike = full[full["month"] == peak]
print(f"\n{peak}: {len(spike):,} sales")
print(spike["property_type"].value_counts().to_string())
print(spike["county"].value_counts().head(5).to_string())

In [ ]:
# Seasonality 

full["month_num"] = full["date"].dt.month

seasonal = full.groupby("month_num").agg(
    n_sales=("price_incl_vat", "size"),
    median=("price_incl_vat", "median"),
)
seasonal["pct_of_annual_avg"] = (
    seasonal["n_sales"] / seasonal["n_sales"].mean() * 100
).round(1)

print(seasonal.round(0).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor(BG)

names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

ax = style_axis(axes[0])
ax.bar(names, seasonal["n_sales"], color=NAVY, alpha=0.8)
ax.set_title("Sales volume by month of year", fontsize=10, fontweight="bold")
ax.set_ylabel("Total sales, 2010-2026")

ax = style_axis(axes[1])
ax.bar(names, seasonal["median"], color=RED, alpha=0.8)
euro(ax)
ax.set_ylim(seasonal["median"].min() * 0.95, seasonal["median"].max() * 1.02)
ax.set_title("Median price by month of year", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
annual["complete_year"] = annual.index < 2026
print(annual[["n_sales", "median", "yoy_pct", "complete_year"]].tail(5).to_string())

In [ ]:
# Does month matter once the trend is removed? 
# express each sale relative to its year's median, then look for
# a residual monthly pattern

year_median = full.groupby("year")["price_incl_vat"].transform("median")
full["price_rel_year"] = full["price_incl_vat"] / year_median

seasonal_adj = full.groupby("month_num")["price_rel_year"].median().round(4)
print("Median price relative to year median, by calendar month:")
print(seasonal_adj.to_string())
print(f"\nSpread: {(seasonal_adj.max() - seasonal_adj.min()) * 100:.1f}%")

Pooled across all years, median price appears flat across the calendar. Expressing each sale relative to its own year's median removes the trend and reveals a genuine seasonal pattern: February is the cheapest month at 0.953 of the year median and October the dearest at 1.029, a spread of 7.6%.

This likely reflects composition rather than pricing - larger family homes transacting through summer and autumn to fit the school year, with smaller and cheaper units clearing in the quiet winter months. Volume and price seasonality are decoupled: December is the busiest month but sits mid-range on price.

Calendar month is retained as a candidate model feature on the strength of the detrended pattern.

## 3. Geographic variation
### 3.1 County level

Median prices in 2025 range from €495,000 in Dublin to €185,000 in Longford, a ratio of 2.7 to 1. Wicklow, Kildare and Meath follow Dublin directly, reflecting the commuter belt.

Dublin also shows by far the widest internal spread, with an interquartile range of €385,000 to €650,000 against Longford's €138,000 to €255,000. Dublin is not only the most expensive county but the most internally varied, which is the central argument for modelling below county level.

In [ ]:
# County-level prices, latest full year 

latest = full[full["year"] == 2025]

county_stats = latest.groupby("county").agg(
    n_sales=("price_incl_vat", "size"),
    median=("price_incl_vat", "median"),
    p25=("price_incl_vat", lambda s: s.quantile(0.25)),
    p75=("price_incl_vat", lambda s: s.quantile(0.75)),
).round(0).sort_values("median", ascending=False)

print(county_stats.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
fig.patch.set_facecolor(BG)
style_axis(ax)

y = np.arange(len(county_stats))
ax.barh(y, county_stats["median"], color=NAVY, alpha=0.85, height=0.7)
ax.errorbar(county_stats["median"], y,
            xerr=[county_stats["median"] - county_stats["p25"],
                  county_stats["p75"] - county_stats["median"]],
            fmt="none", ecolor=RED, elinewidth=1.2, capsize=3, alpha=0.7)

ax.set_yticks(y)
ax.set_yticklabels(county_stats.index, fontsize=9)
ax.invert_yaxis()
euro(ax, axis="x")
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.yaxis.grid(False)
ax.set_xlabel("Median sale price, 2025")
ax.set_title("Median property price by county, 2025\nbars show median, whiskers show interquartile range",
             fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Within-county variation 
# how much price variation does county alone fail to capture?

recent = eircode_era[eircode_era["year"] >= 2024]

rk_stats = (recent.groupby(["rk_county", "routing_key"])
            .agg(n=("price_incl_vat", "size"),
                 median=("price_incl_vat", "median"))
            .reset_index())
rk_stats = rk_stats[rk_stats["n"] >= 50]

spread = (rk_stats.groupby("rk_county")
          .agg(n_keys=("routing_key", "size"),
               lowest=("median", "min"),
               highest=("median", "max"),
               county_median=("median", "median"))
          .round(0))
spread["ratio"] = (spread["highest"] / spread["lowest"]).round(2)
spread = spread[spread["n_keys"] >= 3].sort_values("ratio", ascending=False)

print(spread.to_string())

In [ ]:
# Dublin districts, the clearest case

dub = recent[recent["dublin_district"].notna()]

dub_stats = dub.groupby("dublin_district").agg(
    n=("price_incl_vat", "size"),
    median=("price_incl_vat", "median"),
    p25=("price_incl_vat", lambda s: s.quantile(0.25)),
    p75=("price_incl_vat", lambda s: s.quantile(0.75)),
).round(0)
dub_stats = dub_stats[dub_stats["n"] >= 50].sort_values("median", ascending=False)

print(dub_stats.to_string())
print(f"\nDublin spread: {dub_stats['median'].max() / dub_stats['median'].min():.2f}x")

### 3.2 Below county level

Within-county variation confirms that county is too coarse a geographic unit. Among counties with at least three routing keys of sufficient volume, Dublin's dearest routing key is 2.34 times its cheapest and Cork's is 2.16 times. At the other end, Waterford and Donegal vary by less than 20%.

Dublin postal districts show the pattern most clearly. Dublin 14 has a median of €711,000 against Dublin 10 at €324,500, a ratio of 2.19 to 1. County-level modelling would treat these as identical locations.

The value of routing key granularity therefore varies geographically - substantial in Dublin and Cork, marginal in the smaller counties where transaction volume also limits how finely the data can be cut.

## 4. New builds

New build prices carry a premium over second-hand that has changed markedly over the period. The premium is negligible through 2010 to 2014, then steps to 51% in 2016 and holds above 50% until 2020, before compressing steadily to 27% by 2026.

This tracks supply. New builds fall to 12.4% of transactions in 2013, and the limited output of that period was concentrated in higher-value locations. As construction recovered from 2021 the mix broadened and the premium narrowed.

In [ ]:
# New versus second-hand 

pt = full.groupby(["year", "property_type"])["price_incl_vat"].median().unstack()
pt["premium_pct"] = ((pt["New"] / pt["Second-Hand"] - 1) * 100).round(1)
print(pt.round(0).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor(BG)

ax = style_axis(axes[0])
ax.plot(pt.index, pt["New"], color=GREEN, linewidth=1.8, label="New")
ax.plot(pt.index, pt["Second-Hand"], color=NAVY, linewidth=1.8, label="Second-hand")
euro(ax)
ax.set_title("Median price by property type", fontsize=10, fontweight="bold")
ax.legend(fontsize=9)

ax = style_axis(axes[1])
share = (full.groupby(["year", "property_type"]).size().unstack())
share_pct = share.div(share.sum(axis=1), axis=0) * 100
ax.plot(share_pct.index, share_pct["New"], color=GREEN, linewidth=1.8)
ax.set_ylabel("New builds, % of sales")
ax.set_title("New build share of transactions", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

Comparing new against second-hand within the same routing key tests whether the premium reflects genuine willingness to pay or simply where new builds get constructed. Restricting to routing keys with at least 30 sales of each type in 2024 onward gives a median within-area premium of 25.0%, with an interquartile range of 19.8% to 37.8%.

This is close to the national figure, indicating the premium is genuine rather than a composition effect.

A note on the filter. An earlier version of this analysis required 100 sales per routing key in total rather than per property type. That allowed routing keys where one property type had very few transactions to pass - Dublin 8 showed an apparent new build median of €22,700, derived from a single €20,000 filing. Requiring a minimum count per type rather than in aggregate is essential when comparing group medians, and reduces the usable sample from 121 routing keys to 36.

## 5. Energy efficiency and price

BER data is available only at area level, since the public research extract removes the address fields needed for a property-level join. The correlations below therefore describe areas, not properties.

In [ ]:
# Does area energy efficiency relate to price? 

area_summary = (recent.groupby("join_area")
                .agg(n=("price_incl_vat", "size"),
                     median_price=("price_incl_vat", "median"),
                     ber_rating=("ber_mean_rating", "first"),
                     pct_ab=("ber_pct_a_or_b", "first"),
                     floor_area=("ber_mean_floor_area", "first"),
                     year_built=("ber_median_year_built", "first"))
                .query("n >= 100"))

print(f"Areas: {len(area_summary)}")
print(area_summary.corr(numeric_only=True)["median_price"].round(3).to_string())

In [ ]:
# Is the new build premium composition or genuine? 
# compare new vs second-hand within the same routing key

recent_24 = eircode_era[eircode_era["year"] >= 2024]

prem = (recent_24.groupby(["routing_key", "property_type"])["price_incl_vat"]
        .median().unstack())
prem = prem.join(recent_24.groupby("routing_key").size().rename("n"))
prem = prem[(prem["n"] >= 100) & prem["New"].notna() & prem["Second-Hand"].notna()]
prem["premium_pct"] = ((prem["New"] / prem["Second-Hand"] - 1) * 100).round(1)

print(f"Routing keys with both types: {len(prem)}")
print(f"Median within-area premium: {prem['premium_pct'].median():.1f}%")
print(prem.sort_values("premium_pct")[["n", "New", "Second-Hand", "premium_pct"]].to_string())

In [ ]:
# demonstrate the urban/rural confound directly
area_summary["is_dublin"] = area_summary.index.str.startswith("Dublin")
print(area_summary.groupby("is_dublin")[
    ["median_price", "floor_area", "year_built", "ber_rating"]
].median().round(1).to_string())

Area median price correlates negatively with mean BER rating at -0.216, in the expected direction since lower ratings indicate better efficiency, and positively with the proportion of dwellings rated A or B at 0.231. Both are weak.

More striking are the correlations with mean floor area at -0.420 and median construction year at -0.473, both negative. Areas with larger and newer dwellings have lower prices.

This is an ecological fallacy risk and should not be read as a property-level relationship. Dublin areas have a mean dwelling size of 92 square metres against 127 elsewhere, a median construction year of 1988 against 1999, and a median price of €468,500 against €260,000. The negative correlations reflect settlement patterns, not a discount for space or newness. At property level within an area, floor area would be expected to correlate positively with price.

A further caveat applies to the BER data itself. A BER assessment is required only at point of sale or rental, so the dataset over-represents recently transacted and newly built stock. ber_pct_a_or_b measures the efficiency of assessed dwellings in an area, not of the housing stock as a whole.

In [ ]:
d08 = eircode_era[(eircode_era["routing_key"] == "D08") &
                  (eircode_era["year"] >= 2024) &
                  (eircode_era["property_type"] == "New")]

print(f"D08 new builds 2024+: {len(d08):,}")
print(d08["price_incl_vat"].describe().apply(lambda v: f"{v:,.0f}").to_string())
print(f"\nUnder 50k: {(d08['price_incl_vat'] < 50_000).sum():,}")
print(d08.nsmallest(20, "price_incl_vat")[
    ["date", "address", "price", "price_incl_vat", "vat_exclusive"]].to_string())

In [ ]:
# New build premium, within routing key
# both property types need sufficient volume for a meaningful
# median - filtering on total sales alone allows a single
# transaction to define a category

recent_24 = eircode_era[eircode_era["year"] >= 2024]

counts = (recent_24.groupby(["routing_key", "property_type"])
          .size().unstack(fill_value=0))
meds = (recent_24.groupby(["routing_key", "property_type"])["price_incl_vat"]
        .median().unstack())

MIN_PER_TYPE = 30
ok = (counts["New"] >= MIN_PER_TYPE) & (counts["Second-Hand"] >= MIN_PER_TYPE)

prem = meds[ok].copy()
prem["n_new"] = counts.loc[ok, "New"]
prem["n_sh"] = counts.loc[ok, "Second-Hand"]
prem["premium_pct"] = ((prem["New"] / prem["Second-Hand"] - 1) * 100).round(1)

print(f"Routing keys with >= {MIN_PER_TYPE} of each type: {len(prem)}")
print(f"Median within-area premium: {prem['premium_pct'].median():.1f}%")
print(f"IQR: {prem['premium_pct'].quantile(0.25):.1f}% to {prem['premium_pct'].quantile(0.75):.1f}%")
print()
print(prem.sort_values("premium_pct")[
    ["n_new", "n_sh", "New", "Second-Hand", "premium_pct"]].to_string())

In [ ]:
nominal = recent_24[recent_24["price_incl_vat"] < 50_000]
print(f"Sales under €50k, 2024+: {len(nominal):,} ({len(nominal)/len(recent_24)*100:.2f}%)")
print(nominal["property_type"].value_counts().to_string())
print(f"\nOf which flagged not-full-market: {(nominal['not_full_market']=='Yes').sum():,}")
print(f"\n{nominal.nsmallest(15, 'price_incl_vat')[['date','address','county','price_incl_vat']].to_string()}")

In [ ]:
low = recent_24[recent_24["price_incl_vat"] < 100_000]
print(f"Under €100k: {len(low):,} ({len(low)/len(recent_24)*100:.2f}%)")

print("\nMost common exact prices under €100k:")
print(low["price"].value_counts().head(20).to_string())

print("\nDistribution:")
for b in [20, 25, 30, 40, 50, 60, 75, 100]:
    n = (recent_24["price_incl_vat"] < b * 1000).sum()
    print(f"  under €{b}k: {n:,}")

## 6. Price per square metre

Dividing area median price by mean BER floor area gives an indicative price per square metre. This is an area-level approximation rather than a true per-property ratio, since the two inputs come from different datasets at different levels of aggregation.

In [ ]:
# Approximate price per square metre by area 
# BER floor area is an area-level average, not a per-property
# measure, so this is indicative rather than exact

area_psm = (recent_24.groupby("join_area")
            .agg(n=("price_incl_vat", "size"),
                 median_price=("price_incl_vat", "median"),
                 floor_area=("ber_mean_floor_area", "first"),
                 ber_rating=("ber_mean_rating", "first"),
                 pct_ab=("ber_pct_a_or_b", "first"))
            .query("n >= 200"))

area_psm["price_per_sqm"] = (area_psm["median_price"] / area_psm["floor_area"]).round(0)
area_psm["is_dublin"] = area_psm.index.str.startswith("Dublin")

print(f"Areas: {len(area_psm)}")
print(area_psm.sort_values("price_per_sqm", ascending=False)[
    ["n", "median_price", "floor_area", "price_per_sqm"]].head(20).to_string())
print("\nLowest:")
print(area_psm.sort_values("price_per_sqm")[
    ["n", "median_price", "floor_area", "price_per_sqm"]].head(10).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor(BG)

ax = style_axis(axes[0])
for is_dub, colour, label in [(True, RED, "Dublin district"), (False, NAVY, "County")]:
    sub = area_psm[area_psm["is_dublin"] == is_dub]
    ax.scatter(sub["floor_area"], sub["median_price"],
               s=sub["n"] / 20, color=colour, alpha=0.6, label=label,
               edgecolors="white", linewidth=0.5)
euro(ax)
ax.set_xlabel("Mean dwelling floor area, sq m (BER)")
ax.set_ylabel("Median sale price")
ax.set_title("Price against dwelling size by area\npoint size proportional to sales volume",
             fontsize=10, fontweight="bold")
ax.legend(fontsize=9)

ax = style_axis(axes[1])
for is_dub, colour, label in [(True, RED, "Dublin district"), (False, NAVY, "County")]:
    sub = area_psm[area_psm["is_dublin"] == is_dub]
    ax.scatter(sub["ber_rating"], sub["price_per_sqm"],
               s=sub["n"] / 20, color=colour, alpha=0.6, label=label,
               edgecolors="white", linewidth=0.5)
euro(ax)
ax.set_xlabel("Mean BER rating (lower is more efficient)")
ax.set_ylabel("Price per sq m")
ax.set_title("Energy efficiency against price per sq m",
             fontsize=10, fontweight="bold")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

Price per square metre ranges from €7,439 in Dublin 6 to €1,396 in Donegal - a ratio of 5.3 to 1, considerably wider than the 2.7 to 1 range on headline price. Dublin's higher prices come attached to smaller dwellings, so the gap widens substantially when size is accounted for.

The scatter plots show two largely non-overlapping clusters. Dublin districts occupy 60 to 120 square metres at €340,000 to €710,000; counties occupy 120 to 135 square metres at €185,000 to €420,000. The apparent negative relationship between efficiency and price per square metre is visibly driven by this urban and rural division rather than by efficiency pricing within comparable areas.

## 7. Routing key ranking

The highest and lowest 25 routing keys by median price, restricted to those with at least 100 sales from 2024 onward.

In [ ]:
# Routing key price ranking

rk_recent = (recent_24.groupby("routing_key")
             .agg(n=("price_incl_vat", "size"),
                  median=("price_incl_vat", "median"),
                  county=("rk_county", "first"))
             .query("n >= 100")
             .sort_values("median", ascending=False))

print(f"Routing keys: {len(rk_recent)}")

top_n = 25
combined = pd.concat([rk_recent.head(top_n), rk_recent.tail(top_n)])

fig, ax = plt.subplots(figsize=(10, 12))
fig.patch.set_facecolor(BG)
style_axis(ax)

colours = [RED if i < top_n else NAVY for i in range(len(combined))]
labels = [f"{rk}  ({c})" for rk, c in zip(combined.index, combined["county"])]

y = np.arange(len(combined))
ax.barh(y, combined["median"], color=colours, alpha=0.85, height=0.75)
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis()
euro(ax, axis="x")
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.yaxis.grid(False)
ax.axhline(top_n - 0.5, color="black", linewidth=1.0, linestyle="--", alpha=0.5)
ax.set_xlabel("Median sale price, 2024 onward")
ax.set_title(f"Highest and lowest {top_n} routing keys by median price\n"
             f"minimum 100 sales", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

The top of the ranking is dominated by Dublin, including several routing keys outside the numbered postal districts - A94 Blackrock, A96 Foxrock and A63 Greystones sit among the highest despite carrying non-D codes. The bottom is concentrated in the northwest and midlands, with Donegal, Mayo, Roscommon and Leitrim appearing repeatedly.

The spread across routing keys is roughly 4.5 to 1, wider than the county-level range and confirming routing key as the appropriate geographic unit for modelling.

## 8. Summary

Trend. Prices fell 31% to a 2013 trough and have risen 161% since, with growth settling at 8-10% annually from 2021.

Seasonality. Strong on volume, weak but real on price. Retained as a candidate feature.

Geography. County is too coarse. Within-county variation reaches 2.34 to 1 in Dublin, and routing keys span roughly 4.5 to 1 nationally. Routing key is the modelling unit.

Property type. New builds command a genuine 25% premium within the same area, down from over 50% in the late 2010s.

Energy efficiency. Area-level BER features correlate weakly with price and are confounded by urban and rural settlement patterns. Their value in a model that already controls for location is uncertain and will be tested directly.

Next

Notebook 04 covers modelling: predicting sale price from routing key, property type, sale date and area-level characteristics, with a comparison of linear and gradient boosting approaches.